# Automata Sparsity Benchmarks

This notebook benchmarks the **sparsity** of automata produced by query compilation across existing modules:

- `autstr.algebra`
- `autstr.buildin.presentations.BuechiArithmeticZ` (arithmetic)
- `autstr.graphs`
- `autstr.buildin.presentations.MSO0`

We keep examples at manageable size and report:

- number of states,
- symbol arity,
- alphabet size,
- average exceptions per state,
- exception density over the dense transition table,
- compile time in milliseconds.

In [1]:
import time

from autstr.algebra import FiniteBooleanAlgebras
from autstr.groups import FiniteAbelianGroups
from autstr.graphs import TreeDepthClass, PathWidthClass
from autstr.buildin.presentations import BuechiArithmeticZ, MSO0

In [2]:
def dfa_sparsity_stats(dfa):
    alphabet_size = len(dfa.base_alphabet)
    per_state_dense = alphabet_size ** dfa.symbol_arity
    dense_transitions = dfa.num_states * per_state_dense

    explicit_exceptions = int((dfa.exception_symbols != -1).sum())
    avg_exceptions = explicit_exceptions / dfa.num_states if dfa.num_states else 0.0
    exception_density = (
        explicit_exceptions / dense_transitions if dense_transitions else 0.0
    )

    # Number of explicit exceptions used by the busiest state.
    if dfa.num_states:
        max_exceptions_per_state = int((dfa.exception_symbols != -1).sum(axis=1).max())
    else:
        max_exceptions_per_state = 0

    # Dense table stores one transition per symbol and state.
    # Sparse storage stores one default target per state + explicit exceptions.
    sparse_entries = dfa.num_states + explicit_exceptions
    compression_ratio = dense_transitions / sparse_entries if sparse_entries else 0.0

    return {
        'states': dfa.num_states,
        'arity': dfa.symbol_arity,
        'alphabet': alphabet_size,
        'avg_exceptions_per_state': avg_exceptions,
        'max_exceptions_per_state': max_exceptions_per_state,
        'exception_density': exception_density,
        'compression_ratio': compression_ratio,
        'accepting_states': int(dfa.is_accepting.sum()),
    }


def run_uniform_bench(module_name, class_name, evaluator, queries):
    rows = []
    for query_name, phi in queries:
        t0 = time.perf_counter()
        dfa, _ = evaluator(phi)
        dt_ms = (time.perf_counter() - t0) * 1000

        stats = dfa_sparsity_stats(dfa)
        rows.append({
            'module': module_name,
            'class': class_name,
            'query': query_name,
            **stats,
            'compile_ms': dt_ms,
        })
    return rows


def run_presentation_bench(module_name, class_name, presentation, queries):
    rows = []
    for query_name, phi in queries:
        t0 = time.perf_counter()
        dfa = presentation.evaluate(phi)
        dt_ms = (time.perf_counter() - t0) * 1000

        stats = dfa_sparsity_stats(dfa)
        rows.append({
            'module': module_name,
            'class': class_name,
            'query': query_name,
            **stats,
            'compile_ms': dt_ms,
        })
    return rows


def print_table(rows, columns):
    if not rows:
        print('no rows')
        return

    widths = {}
    for col in columns:
        widths[col] = len(col)
        for row in rows:
            widths[col] = max(widths[col], len(str(row[col])))

    header = ' | '.join(col.ljust(widths[col]) for col in columns)
    sep = '-+-'.join('-' * widths[col] for col in columns)
    print(header)
    print(sep)
    for row in rows:
        print(' | '.join(str(row[col]).ljust(widths[col]) for col in columns))

## Query sets

All formulas are intentionally modest to keep compile times practical.

In [3]:
algebra_queries_boolean = [
    ('atom', 'Atom(x)'),
    ('proper-element', 'exists x.(exists y.(Compl(x,y) and (not Leq(x,y))))'),
    ('two-distinct-atoms', 'exists x.(exists y.(Atom(x) and Atom(y) and (not Leq(x,y)) and (not Leq(y,x))))'),
]

algebra_queries_abelian = [
    ('addition', 'A(x,y,z)'),
    ('two-torsion', 'exists x.((not A(x,x,x)) and (exists z.(A(x,x,z) and A(z,z,z))))'),
    ('commutativity', 'all x.(all y.(all z.((not A(x,y,z)) or A(y,x,z))))'),
]

arithmetic_queries = [
    ('addition', 'A(x,y,z)'),
    ('nonnegative-lt', 'Lt(x,y)'),
    ('existential-shift', 'exists z.(A(x,z,y))'),
    ('triangle-order', 'exists z.(Lt(x,z) and Lt(z,y))'),
]

graph_queries = [
    ('has-edge', 'exists x.(exists y.(E(x,y)))'),
    ('singleton-exists', 'exists x.(Sing(x))'),
    ('triangle-free', 'all x.(all y.(all z.((not E(x,y)) or (not E(y,z)) or (not E(z,x)))))'),
    ('bipartite-mso', 'exists c.(all x.(all y.((not E(x,y)) or ((Subset(x,c) and (not Subset(y,c))) or ((not Subset(x,c)) and Subset(y,c))))))'),
]

mso0_queries = [
    ('subset', 'Subset(x,y)'),
    ('singleton', 'Sing(x)'),
    ('successor', 'Succ(x,y)'),
    ('some-successor', 'exists x.(exists y.(Succ(x,y)))'),
    ('strict-singleton-order', 'exists x.(exists y.(Sing(x) and Sing(y) and Lt_sing(x,y)))'),
]

In [4]:
rows = []

# algebra
ba = FiniteBooleanAlgebras()
ab = FiniteAbelianGroups()
rows += run_uniform_bench('algebra', 'FiniteBooleanAlgebras', ba.evaluate, algebra_queries_boolean)
rows += run_uniform_bench('algebra', 'FiniteAbelianGroups', ab.evaluate, algebra_queries_abelian)

# arithmetic
arith = BuechiArithmeticZ()
rows += run_presentation_bench('arithmetic', 'BuechiArithmeticZ', arith, arithmetic_queries)

# graphs
# Slightly complex formulas, but bounds kept manageable for notebook runs.
td = TreeDepthClass(3)
pw = PathWidthClass(2)
rows += run_uniform_bench('graphs', 'TreeDepthClass(d=3)', td.evaluate, graph_queries)
rows += run_uniform_bench('graphs', 'PathWidthClass(w=2)', pw.evaluate, graph_queries)

# mso0
mso0 = MSO0()
rows += run_presentation_bench('mso0', 'MSO0', mso0, mso0_queries)

rows = sorted(rows, key=lambda r: (r['module'], r['class'], r['query']))

# Keep a numeric copy for summary statistics.
rows_numeric = [dict(r) for r in rows]

for row in rows:
    row['avg_exceptions_per_state'] = f"{row['avg_exceptions_per_state']:.2f}"
    row['exception_density'] = f"{row['exception_density']:.6f}"
    row['compile_ms'] = f"{row['compile_ms']:.2f}"

columns = [
    'module', 'class', 'query',
    'states', 'accepting_states',
    'arity', 'alphabet',
    'avg_exceptions_per_state', 'exception_density',
    'compile_ms',
]

print_table(rows, columns)

module     | class                 | query                  | states | accepting_states | arity | alphabet | avg_exceptions_per_state | exception_density | compile_ms
-----------+-----------------------+------------------------+--------+------------------+-------+----------+--------------------------+-------------------+-----------
algebra    | FiniteAbelianGroups   | addition               | 18     | 3                | 4     | 4        | 7.72                     | 0.030165          | 19.96     
algebra    | FiniteAbelianGroups   | commutativity          | 6      | 3                | 1     | 4        | 1.83                     | 0.458333          | 14.98     
algebra    | FiniteAbelianGroups   | two-torsion            | 9      | 3                | 1     | 4        | 2.00                     | 0.500000          | 22.86     
algebra    | FiniteBooleanAlgebras | atom                   | 4      | 2                | 2     | 3        | 1.25                     | 0.138889          | 2.94     

## Interpretation

- Lower `exception_density` means the sparse DFA representation is exploiting more default transitions.
- `max_exceptions_per_state` highlights the worst-case state complexity in sparse form.
- `compression_ratio` estimates dense-to-sparse transition storage gain.
- `compile_ms` is query compilation time in this environment; rerun to compare machines/settings.
- The next cell reports mean exception density and mean compression ratio overall and by module/class.

In [5]:
def summarize_density_and_compression(rows_numeric):
    overall_density = sum(r['exception_density'] for r in rows_numeric) / len(rows_numeric)
    overall_compression = sum(r['compression_ratio'] for r in rows_numeric) / len(rows_numeric)

    by_module = {}
    by_class = {}
    for r in rows_numeric:
        by_module.setdefault(r['module'], []).append(r)
        by_class.setdefault((r['module'], r['class']), []).append(r)

    module_rows = [
        {
            'group': 'module',
            'name': m,
            'mean_exception_density': f"{sum(x['exception_density'] for x in vals) / len(vals):.6f}",
            'mean_compression_ratio': f"{sum(x['compression_ratio'] for x in vals) / len(vals):.2f}",
            'queries': len(vals),
        }
        for m, vals in sorted(by_module.items())
    ]

    class_rows = [
        {
            'group': 'class',
            'name': f"{m} / {c}",
            'mean_exception_density': f"{sum(x['exception_density'] for x in vals) / len(vals):.6f}",
            'mean_compression_ratio': f"{sum(x['compression_ratio'] for x in vals) / len(vals):.2f}",
            'queries': len(vals),
        }
        for (m, c), vals in sorted(by_class.items())
    ]

    print(f"overall mean exception density: {overall_density:.6f}")
    print(f"overall mean compression ratio: {overall_compression:.2f}")
    print()
    print_table(
        module_rows + class_rows,
        ['group', 'name', 'mean_exception_density', 'mean_compression_ratio', 'queries']
    )


summarize_density_and_compression(rows_numeric)

overall mean exception density: 0.304542
overall mean compression ratio: 3.41

group  | name                            | mean_exception_density | mean_compression_ratio | queries
-------+---------------------------------+------------------------+------------------------+--------
module | algebra                         | 0.299009               | 6.52                   | 6      
module | arithmetic                      | 0.386752               | 2.49                   | 4      
module | graphs                          | 0.363480               | 2.36                   | 8      
module | mso0                            | 0.151111               | 2.07                   | 5      
class  | algebra / FiniteAbelianGroups   | 0.329499               | 10.70                  | 3      
class  | algebra / FiniteBooleanAlgebras | 0.268519               | 2.33                   | 3      
class  | arithmetic / BuechiArithmeticZ  | 0.386752               | 2.49                   | 4      
class  | gra